# 025 — Razonamiento con incertidumbre

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Soluciones explicadas

**E1.** `P(Engancha=sí) = 0.108+0.072+0.016+0.144 = 0.340`. Con caries: `0.108+0.072 = 0.180`. Posterior: `0.180/0.340 ≈ 0.529`. El gancho eleva la creencia de 0.20 a 0.53 — menos que el dolor (0.60) porque el gancho también se engancha sin caries con probabilidad no trivial.

**E2.** `P(Caries=no)=0.8`. `P(Dolor=sí|¬C)=0.080/0.8=0.10`; `P(Engancha=sí|¬C)=0.160/0.8=0.20`; conjunta: `0.016/0.8=0.02 = 0.10·0.20`. ✔ Se cumple: dado el estado de la caries, dolor y gancho son independientes — es la estructura de causa común que explota la clase 027.

**E3.** `P(Dolor=sí|Caries=sí)=0.120/0.200=0.60` y `P(Caries=sí|Dolor=sí)=0.120/0.200=0.60`. En esta tabla coinciden *numéricamente* porque `P(Caries=sí)=P(Dolor=sí)=0.2`; son cantidades distintas que solo empatan cuando los marginales empatan (forma odds de Bayes). Cambiar el prior rompe la coincidencia: no son la misma cantidad.

**E4.** Cambian los valores muestreados/derivados de la semilla; son estructurales `kind`, `seed`, `evidence` y `limitations` — el contrato del laboratorio.


In [ ]:
result = run_lab("probability", seed=25)
assert result["kind"] == "probability"
assert result["evidence"]
show(result)


In [ ]:
conjunta = {
    ("si","si","si"): 0.108, ("si","si","no"): 0.012,
    ("si","no","si"): 0.072, ("si","no","no"): 0.008,
    ("no","si","si"): 0.016, ("no","si","no"): 0.064,
    ("no","no","si"): 0.144, ("no","no","no"): 0.576,
}  # claves: (caries, dolor, engancha)

def P(pred):
    return sum(p for k, p in conjunta.items() if pred(*k))

pE = P(lambda c,d,e: e=="si"); pCE = P(lambda c,d,e: e=="si" and c=="si")
print("E1:", round(pCE/pE, 3))
pnC = P(lambda c,d,e: c=="no")
pD = P(lambda c,d,e: c=="no" and d=="si")/pnC
pEn = P(lambda c,d,e: c=="no" and e=="si")/pnC
pDE = P(lambda c,d,e: c=="no" and d=="si" and e=="si")/pnC
print("E2:", round(pD,3), round(pEn,3), round(pDE,3), "==", round(pD*pEn,3))
for s in (25, 99):
    r = run_lab("probability", seed=s)
    assert r["kind"] == "probability" and r["evidence"]
print("E4: contrato estable en ambas semillas")


## Reflexión

1. En la salida del laboratorio, ¿qué número corresponde a un *prior* y cuál a un *posterior*? ¿Qué evidencia separa a uno del otro?
2. Si duplicáramos el número de variables del dominio, ¿por qué la inferencia por enumeración se vuelve inviable y qué propiedad (vista en la clase 027) lo remedia?
3. Con `P(caries | evidencia) = 0.6`, ¿basta para decidir tratar? ¿Qué elemento adicional exige la teoría de la decisión (clase 030)?
